<a href="https://colab.research.google.com/github/129Ashish/129Ashish11/blob/main/7.Fashion_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries Required

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [3]:
torch.manual_seed(42)

In [4]:
#checking GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device : {device}")

Using device : cuda


# Data Collection

In [16]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [17]:
transform = transforms.ToTensor()

In [18]:
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

In [21]:
print(f"shape of an individual image in the train dataset: {train_dataset[0][0].shape}")

shape of an individual image in the train dataset: torch.Size([1, 28, 28])


In [28]:
print(len(train_dataset))

60000


In [29]:
print(len(test_dataset))

10000


# Dataset Class Objects

In [30]:


#create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels = torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index] , self.labels[index]

# DataLoader Objects

In [31]:
# create a train and test dataloader objects using the DataLoader Classes
train_loader = DataLoader(train_dataset ,batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=128,shuffle=False)

# Defining the Model

In [33]:
#Define NN class
class MyNN(nn.Module):

  def __init__(self,num_features):
    super().__init__()
    self.model=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10),
        # nn.LogSoftmax(dim=1) no need to define softmx externally because in entropy loss function it is built in defined there alrady
    )

  def forward(self,x):
    return self.model(x)


# Important Parameters

In [39]:

#set learning rate and epochs
learning_rate = 0.1
epochs=100
#loss function
criterion = nn.CrossEntropyLoss()
#defining model
model = MyNN(784)
#optimizer
optimizer=optim.SGD(model.parameters(),lr=learning_rate)

In [40]:
28*28 #num_features is the flattened input size to the network i.e. 28*28 = 784

784

In [42]:
len(train_loader) # nu. of batches = 469

469

# Training Pipeline

In [45]:
#training loop
for epoch in range(epochs):
  total_epoch_loss=0
  for batch_features,batch_labels in train_loader:
    # Flatten the image data
    batch_features = batch_features.view(-1, 28 * 28)

    #forward pass
    outputs=model(batch_features)
    #loss calculate
    loss=criterion(outputs,batch_labels)
    #gradient zero
    optimizer.zero_grad()
    #back propogation
    loss.backward()
    #upgrade gradients
    optimizer.step()

    total_epoch_loss =total_epoch_loss+ loss.item()

  avg_loss=total_epoch_loss/len(train_loader) #avg epoch loss
  print(f"epoch : {epoch+1}, Loss : {avg_loss}")

epoch : 1, Loss : 0.40641502263957757
epoch : 2, Loss : 0.3808709505969273
epoch : 3, Loss : 0.36178431420056806
epoch : 4, Loss : 0.348546589202464
epoch : 5, Loss : 0.3353997176326414
epoch : 6, Loss : 0.3231859233524245
epoch : 7, Loss : 0.3150471015207803
epoch : 8, Loss : 0.3041025057339719
epoch : 9, Loss : 0.2967119299526662
epoch : 10, Loss : 0.28918274163183116
epoch : 11, Loss : 0.2830478057805409
epoch : 12, Loss : 0.2740346815730971
epoch : 13, Loss : 0.26863109619060815
epoch : 14, Loss : 0.26563974890881764
epoch : 15, Loss : 0.2593865308489627
epoch : 16, Loss : 0.2542843175436388
epoch : 17, Loss : 0.24869578863893235
epoch : 18, Loss : 0.24358842917469772
epoch : 19, Loss : 0.23973466662455722
epoch : 20, Loss : 0.2351070783539876
epoch : 21, Loss : 0.2317565922766352
epoch : 22, Loss : 0.22740552585516402
epoch : 23, Loss : 0.222539528632469
epoch : 24, Loss : 0.21744787808992208
epoch : 25, Loss : 0.21689232097251582
epoch : 26, Loss : 0.2128404582709646
epoch : 27, 

# Evaluation

In [46]:
#set model to eval mode
model.eval()


MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [49]:
#evaluation code
total=0
correct=0
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    # Flatten the image data
    batch_features = batch_features.view(-1, 28 * 28)

    outputs=model(batch_features)
    _,predicted=torch.max(outputs.data,1)
    total+=batch_labels.shape[0]
    correct+=(predicted==batch_labels).sum().item()
  print(f"Accuracy : {correct/total*100:.3f}")

Accuracy : 88.150


# NOW doing the same by creating a CNN

## Define the cnn model architecture


In [22]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        # Calculate the size of the flattened layer after two pooling layers
        # Input image size is 28x28.
        # After first conv (3x3 kernel, 1 padding): 28 + 2*1 - 3 + 1 = 28
        # After first pooling (2x2 kernel, 2 stride): 28 / 2 = 14
        # After second conv (3x3 kernel, 1 padding): 14 + 2*1 - 3 + 1 = 14
        # After second pooling (2x2 kernel, 2 stride): 14 / 2 = 7
        # The number of output channels is 32, so the flattened size is 32 * 7 * 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10) # 10 classes for FashionMNIST

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7) # Flatten the tensor
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Define the loss function and optimizer


In [23]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Create dataloader objects


In [24]:
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Implement the training loop


In [25]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/100:.4f}')
            running_loss = 0.0

Epoch [1/10], Step [100/469], Loss: 0.9707
Epoch [1/10], Step [200/469], Loss: 0.5640
Epoch [1/10], Step [300/469], Loss: 0.4919
Epoch [1/10], Step [400/469], Loss: 0.4381
Epoch [2/10], Step [100/469], Loss: 0.3979
Epoch [2/10], Step [200/469], Loss: 0.3740
Epoch [2/10], Step [300/469], Loss: 0.3684
Epoch [2/10], Step [400/469], Loss: 0.3610
Epoch [3/10], Step [100/469], Loss: 0.3359
Epoch [3/10], Step [200/469], Loss: 0.3311
Epoch [3/10], Step [300/469], Loss: 0.3168
Epoch [3/10], Step [400/469], Loss: 0.3046
Epoch [4/10], Step [100/469], Loss: 0.2901
Epoch [4/10], Step [200/469], Loss: 0.2977
Epoch [4/10], Step [300/469], Loss: 0.2838
Epoch [4/10], Step [400/469], Loss: 0.2821
Epoch [5/10], Step [100/469], Loss: 0.2761
Epoch [5/10], Step [200/469], Loss: 0.2671
Epoch [5/10], Step [300/469], Loss: 0.2676
Epoch [5/10], Step [400/469], Loss: 0.2597
Epoch [6/10], Step [100/469], Loss: 0.2563
Epoch [6/10], Step [200/469], Loss: 0.2493
Epoch [6/10], Step [300/469], Loss: 0.2510
Epoch [6/10

## Implement the evaluation function


In [26]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy

## Evaluate the trained model


In [27]:
test_accuracy = evaluate_model(model, test_loader, device)
print(f"Model accuracy on the test set: {test_accuracy:.2f}%")

Model accuracy on the test set: 91.21%
